In [ ]:
import sys, os
# Imports of vbn_utils, ccf_utils, decoding_utils, analysis_utils,
# notebook_utils, vbn_4day_utils resolve via vbn_code/utilities/.
_UTILS_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', 'utilities'))
if _UTILS_DIR not in sys.path:
    sys.path.insert(0, _UTILS_DIR)

import h5py
import pandas as pd
import numpy as np
import os
from pathlib import Path
from matplotlib import pyplot as plt
from matplotlib.colors import TwoSlopeNorm, Normalize
import matplotlib.gridspec as gridspec
from matplotlib.patches import Rectangle, Polygon
from matplotlib.collections import PolyCollection
from sklearn.metrics import roc_curve, auc
import scipy.stats
import pickle
import warnings
warnings.filterwarnings("ignore")
import vbn_utils
import decoding_utils as du
from analysis_utils import exponential_convolve
from analysis_utils import multiple_comparisons
import ccf_utils
from vbn_utils import make_iterable, get_unit_ids, formatFigure, mean_sem_plot
%matplotlib inline

In [ ]:
from notebook_utils import (
    get_mod_index,
    get_mod_index_norm,
    normalize_trace_pair,
    get_nov_mod_index_norm,
    get_nov_mod_index_norm_bootstrap,
    bootstrapped_diff_ci,
    comparison_matrix,
    multiple_comparisons
)

In [ ]:
high_res = True
if high_res:
    plt.rcParams['figure.dpi'] = 300
    plt.rcParams['savefig.dpi'] = 300
    plt.rcParams['font.size'] = 12
    plt.rcParams['pdf.fonttype'] = 42
    
    plt.rcParams['figure.facecolor'] = 'white'
    plt.rcParams['axes.facecolor'] = 'white'
    plt.rcParams['savefig.facecolor'] = 'white'  # affects clipboard copy too
    plt.rcParams['savefig.transparent'] = False

## Data loading

In [ ]:
#Paths to all of the useful supplemental tables and tensors
active_tensor_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/vbnAllUnitSpikeTensor.hdf5"
passive_tensor_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/vbnAllUnitSpikeTensor_passive.hdf5"
#opto_tensor_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/vbn_opto_tensor_unit_grouped.hdf5"

stim_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/master_stim_table_no_filter.csv"
unit_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/master_units_with_responsiveness.csv"

sessions_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/vbn_s3_cache/visual-behavior-neuropixels-0.5.0/project_metadata/ecephys_sessions.csv"

In [ ]:
units = pd.read_csv(unit_table_file)
units['cortical_layer'] = units['cortical_layer'].replace('3-Feb','2/3') # necessary since 2/3 sometimes gets incorrectly reformatted as a date

stim_table = pd.read_csv(stim_table_file)
stim_table = stim_table.drop(columns='Unnamed: 0')

active_tensor = h5py.File(active_tensor_file)

sessions_table = pd.read_csv(sessions_table_file)

In [ ]:
new_clusters = pd.read_csv("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/cluster_labels_122024.csv")
units = units.merge(new_clusters[['unit_id', 'cluster_labels_new_weak_cluster_12', 'used_in_clustering']], on='unit_id', how='left')
units.rename(columns={'cluster_labels_new_weak_cluster_12': 'cluster_labels_new', 'used_in_clustering': 'used_in_new_clustering'}, inplace=True)

In [ ]:
structure_tree = pd.read_csv("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/ccf_structure_tree_2017.csv")

In [ ]:
stim_table['is_shared'] = stim_table['image_name'].isin(['im083_r', 'im111_r'])

## Compute flash data

### Active tensor (if pre-computed, load below)

In [ ]:
import warnings
import tqdm

warnings.filterwarnings("ignore")

unit_filter = du.apply_unit_quality_filter(units)
unit_ids = units.loc[unit_filter]['unit_id'].values

flash_data = {uid: {cond: {stims: [] for stims in ['change', 'prechange','shared_nonchange', 'nonshared_nonchange']} for cond in ['active', 'passive']} for uid in unit_ids}

session_list = [str(sess) for sess in np.unique(units.loc[unit_filter]['ecephys_session_id'])]
for cond, tensor_to_use in zip(['active', 'passive'], [active_tensor_file, passive_tensor_file]):

    #for shared images
    shared_data, nonshared_data, unitIDs = vbn_utils.change_prechange_matched_psth(tensor_to_use, 
                                stim_table, 
                                session_list, 
                                unit_ids, 
                                baseline_length=50, 
                                resp_window_length=750,
                                comparison='shared_nonshared')
    for cdata, predata, uid in zip(np.concatenate(shared_data), np.concatenate(nonshared_data), np.concatenate(unitIDs)):
        flash_data[uid][cond]['shared_nonchange'] = cdata
        flash_data[uid][cond]['nonshared_nonchange'] = predata

    #for non shared images
    change_data, prechange_data, unitIDs = vbn_utils.change_prechange_matched_psth(tensor_to_use, 
                                stim_table, 
                                session_list, 
                                unit_ids, 
                                baseline_length=50, 
                                resp_window_length=750,
                                comparison='change_prechange')
    for cdata, predata, uid in zip(np.concatenate(change_data), np.concatenate(prechange_data), np.concatenate(unitIDs)):
        flash_data[uid][cond]['change'] = cdata
        flash_data[uid][cond]['prechange'] = predata
    

    omission_data, unitIDs = vbn_utils.unit_averaged_psth(tensor_to_use, stim_table, session_list, unit_ids, 
                                *['engaged', 'omitted'], baseline_length=50, resp_window_length=750,)
    for odata, uid in zip(np.concatenate(omission_data), np.concatenate(unitIDs)):
        flash_data[uid][cond]['omission'] = odata


### Passive tensor (if pre-computed, load below)

In [ ]:
import warnings
import tqdm

warnings.filterwarnings("ignore")

unit_filter = du.apply_unit_quality_filter(units)
unit_ids = units.loc[unit_filter]['unit_id'].values

trial_data = {uid: {cond: {stims: [] for stims in ['change', 'prechange']} for cond in ['active', 'passive']} for uid in unit_ids}

session_list = [str(sess) for sess in np.unique(units.loc[unit_filter]['ecephys_session_id'])]
for cond, tensor_to_use in zip(['active', 'passive'], [active_tensor_file, passive_tensor_file]):


    change_data, prechange_data, unitIDs = vbn_utils.change_prechange_matched_unit_response_over_trials(tensor_to_use, stim_table, session_list, 
                                unit_ids, baseline_length=50, resp_window_length=750, 
                                response_slice=slice(20,120), baseline_subtract=True)
    for cdatas, predatas, uids in zip(change_data, prechange_data, unitIDs):
        for cdata, predata, uid in zip(cdatas, predatas, uids):
            trial_data[uid][cond]['change'] = cdata
            trial_data[uid][cond]['prechange'] = predata

In [ ]:
import pickle

with open("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/change_prechange_responses_active_passive_by_unitid.pkl", 'wb') as file:
    pickle.dump(flash_data, file)

# with open("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/change_prechange_trial_by_trial_responses_active_passive_by_unitid.pkl", 'wb') as file:
#     pickle.dump(trial_data, file)

### Load pre-computed flash data

In [ ]:
import pickle

with open("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/change_prechange_responses_active_passive_by_unitid.pkl", 'rb') as file:
    flash_data = pickle.load(file)

with open("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/change_prechange_trial_by_trial_responses_active_passive_by_unitid.pkl", 'rb') as file:
    trial_data = pickle.load(file)

### Identify sessions without significant baseline drift

In [ ]:
import warnings
warnings.filterwarnings("ignore")

unit_filter = du.getUnitsInRegion(units, 'VISall', rs=True) & du.apply_unit_quality_filter(units)# & (units['responds_to_at_least_one_nonchange_image']) #& du.get_units_in_cluster(units, *np.arange(6,14))
session_list = list(active_tensor.keys())

unit_ids = units.loc[unit_filter]['unit_id'].values

            
stim_filter = ['stimulus_presentations_id>0']#, f'session_change_image_counts=={change_image_count}']

baselines, sessionIds = vbn_utils.unit_averaged_psth(active_tensor_file, 
                            stim_table, 
                            session_list, 
                            unit_ids, 
                            *stim_filter, 
                            baseline_length=50, 
                            resp_window_length=10,
                            just_get_baselines=True)


In [ ]:
sessionIds = [sessionIds[ib] for ib, b in enumerate(baselines) if len(b)>0]
baselines = [baselines[ib] for ib, b in enumerate(baselines) if len(b)>0]

In [ ]:
delta_baseline = [b[-1] - b[0] for b in baselines]
low_baseline_drift_sessions = [sessionIds[id] for id, d in enumerate(delta_baseline) if ~np.isnan(d) and d < 0.001]
low_baseline_drift_sessions

In [ ]:
low_baseline_drift_sessions = [int(s) for s in low_baseline_drift_sessions]
low_baseline_drift_units = units[units['ecephys_session_id'].isin(low_baseline_drift_sessions)]
len(low_baseline_drift_units), len(units)

In [ ]:
base_slice = slice(0, 50)
base_sub = lambda x: x - x[:, base_slice].mean(axis=1)[:, None]
no_base_sub = lambda x: x

## Figure-generation functions

In [ ]:
# Pure utility helpers (also available from notebook_utils)
from numba import njit

@njit(cache=True, fastmath=True)
def get_mod_index_norm_njit(vals1, vals2):
    """
    Compute (vals1 - vals2) / max(abs(vals1), abs(vals2)) elementwise.
    """
    n = vals1.size
    out = np.empty(n, dtype=np.float64)
    for i in range(n):
        a = vals1[i]
        b = vals2[i]
        aa = abs(a)
        bb = abs(b)
        denom = aa if aa >= bb else bb
        if denom == 0.0:
            out[i] = 0.0
        else:
            out[i] = (a - b) / denom
    return out


def get_norm_pop_response(units_to_use, state1, state2, flash1, flash2, norm_slice):
    """Compute cross-normalised population responses for two conditions.
    Closes over base_sub, flash_data, exponential_convolve, normalize_trace_pair."""
    resps1 = base_sub(np.array([flash_data[u][state1][flash1] for u in units_to_use]))
    resps2 = base_sub(np.array([flash_data[u][state2][flash2] for u in units_to_use]))
    resps1 = np.array([exponential_convolve(r, 3, symmetrical=True) for r in resps1]) * 1000
    resps2 = np.array([exponential_convolve(r, 3, symmetrical=True) for r in resps2]) * 1000
    resps1_crossnorm = np.array([normalize_trace_pair(r1, r2, norm_slice=norm_slice) for r1, r2 in zip(resps1, resps2)])
    resps2_crossnorm = np.array([normalize_trace_pair(r1, r2, norm_slice=norm_slice) for r1, r2 in zip(resps2, resps1)])
    resps1_crossnorm[~np.isfinite(resps1_crossnorm)] = np.nan
    resps2_crossnorm[~np.isfinite(resps2_crossnorm)] = np.nan
    return resps1_crossnorm, resps2_crossnorm

In [ ]:
plt.rcParams.update({'font.size':16})
def make_psth_figure(areas, layers, cell_types, clusters, experience='all',
                            comparison='change', flashes=['prechange', 'change'], states=['active', 'passive'], 
                            resp_slice=slice(70,150), display_slice=slice(50, 170), norm=True, split=True, 
                            use_ccf_color=False, smooth_kernel_width=3, unit_subsample=units, base_sub_func = base_sub,
                            flash_data=flash_data):
    
    total_splits = np.prod([len(make_iterable(split)) for split in [areas, layers, cell_types]])
    figsize = (8, 4*total_splits) if split else (5, 4*total_splits) 
    fig = plt.figure(figsize=figsize)
    if comparison=='change':
        numcols = len(states)
    if comparison=='state':
        numcols = len(flashes)
    
    num_cols = 2 if split else 1
    gs = gridspec.GridSpec(total_splits, num_cols, figure=fig)
    axes = []
    if comparison=='change':
        contexts = states
    elif comparison=='state':
        contexts = flashes

    for icontext, context in enumerate(contexts):
        
            axcol = icontext if split else 0
            counter = 0
            labels = []
            for ia, area in enumerate(make_iterable(areas)):
                colors = [ccf_utils.get_area_color(area, structure_tree)]*2 if use_ccf_color else ['k', 'darkorange']
                for il, layer in enumerate(make_iterable(layers)):
                    for ic, cell_type in enumerate(make_iterable(cell_types)):
                        
                        axrow = counter
                        units_to_use = get_unit_ids(unit_subsample, area, layers=layer, cell_types=cell_type, 
                                            clusters=clusters, experience=experience, clustering='new')

                        if comparison=='change':
                            if norm:
                                cresps_crossnorm, presps_crossnorm = get_norm_pop_response(units_to_use, context, context, 'change', 'prechange', norm_slice=resp_slice)
                            else:
                                cresps_crossnorm = base_sub_func(np.array([flash_data[u][context]['change'] for u in units_to_use]))
                                presps_crossnorm = base_sub_func(np.array([flash_data[u][context]['prechange'] for u in units_to_use]))
                                
                                cresps_crossnorm = np.array([exponential_convolve(r, smooth_kernel_width, symmetrical=True) for r in cresps_crossnorm])
                                presps_crossnorm = np.array([exponential_convolve(r, smooth_kernel_width, symmetrical=True) for r in presps_crossnorm])

                        else:
                            if norm:
                                cresps_crossnorm, presps_crossnorm = get_norm_pop_response(units_to_use, 'active', 'passive', context, context, norm_slice=resp_slice)
                            else:
                                cresps_crossnorm = base_sub_func(np.array([flash_data[u]['active'][context] for u in units_to_use]))
                                presps_crossnorm = base_sub_func(np.array([flash_data[u]['passive'][context] for u in units_to_use]))
                                
                                cresps_crossnorm = np.array([exponential_convolve(r, smooth_kernel_width, symmetrical=True) for r in cresps_crossnorm])
                                presps_crossnorm = np.array([exponential_convolve(r, smooth_kernel_width, symmetrical=True) for r in presps_crossnorm])
                        
                        if not split and icontext==0:
                            ax = fig.add_subplot(gs[axrow, axcol])
                        elif split:
                            ax = fig.add_subplot(gs[axrow, axcol])
                        else:
                            ax = fig.get_axes()[axrow]

                        line_style = ['-', '--'][icontext] if not split else '-'
                        alpha = [1, 0.5][icontext] if not split else 1


                        vbn_utils.mean_sem_plot(cresps_crossnorm[:, display_slice]*1000, ax, color=colors[0], ls=line_style, alpha=alpha)
                        if split or icontext==0:
                            vbn_utils.mean_sem_plot(presps_crossnorm[:, display_slice]*1000, ax, color=colors[1], ls=line_style, alpha=1,)
                        # ax.axvspan(20, 100, ymin=0, ymax=0.05, color='k', alpha=0.3, lw=0)
                        counter += 1
                        layer_str = ' '+layer if layer != 'all' else ''
                        cell_type_str = ' '+cell_type if cell_type != 'all' else ''
                        labels.append(f'{area}{layer_str}{cell_type_str}')
                        # ax.set_title(f'{area}{layer_str}{cell_type_str} {comparison} mod {context} n:{len(cresps_crossnorm)}')
                        ax.set_title(f'{area}{layer_str}{cell_type_str}, n={len(cresps_crossnorm)}')
                        ax.tick_params(axis='both', labelsize=20)
                        vbn_utils.formatFigure(fig, ax)
                        ax.spines['left'].set_bounds(0, 5)
                        ax.set_yticks([0,5])
                        ax.spines['bottom'].set_bounds(20, 100)
                        ax.set_xticks([20,100])
                        for spine in ax.spines.values():
                            spine.set_linewidth(1.5)
                        ax.tick_params(width=1.5)
                        
    plt.tight_layout()

def make_modulation_figure(areas, layers, cell_types, clusters, metrics=['mod_index', 'mod_index_norm', 'auc', 'ols'],
                            comparison='change', flashes=['change', 'prechange',], states=['active', 'passive'], 
                            plot_comparison_mat=False, return_vals = False, num_iterations=1000, base_sub_func=base_sub,
                            response_summary=response_summary, unit_subsample=units):
    fig_width = 10
    fig_height = 3*len(metrics)
    fig = plt.figure(figsize=(fig_width, fig_height))
    if comparison=='change':
        numcols = len(states)
    if comparison=='state':
        numcols = len(flashes)
    
    gs = gridspec.GridSpec(len(metrics), numcols, figure=fig)
    axes = []

    if comparison=='change':
        contexts = states
    elif comparison=='state':
        contexts = flashes

    ols_keys = ['C(flash)[T.1]', 'C(flash)[T.1]:C(state)[T.1]'] if comparison=='change' else ['C(state)[T.1]', 'C(flash)[T.1]:C(state)[T.1]'] 
    vals_to_return = {c: {m:{} for m in metrics} for c in contexts}
    for icontext, context in enumerate(contexts):
        for imetric, metric in enumerate(metrics):
            ax = fig.add_subplot(gs[imetric, icontext])
            axes.append(ax)
            if metric=='ols':
                title_add = ols_keys[icontext]
            else:
                title_add = context
            ax.set_title(f'{metric} {title_add}')

            if plot_comparison_mat:
                fig_comp, ax_comp = plt.subplots()
                fig_comp.set_size_inches(10, 10)
                fig_comp.suptitle(f'{metric} {context}')

            mods = []
            all_aucs = []
            counter = 0
            labels = []
            all_vals = []
            for ia, area in enumerate(make_iterable(areas)):
                for il, layer in enumerate(make_iterable(layers)):
                    for ic, cell_type in enumerate(make_iterable(cell_types)):
                        color = ccf_utils.get_area_color(area, structure_tree)
                        units_to_use = get_unit_ids(unit_subsample, area, cell_types=cell_type, layers=layer, clusters=clusters,
                                        clustering='new')
                        
                        if comparison == 'change':
                            cresps = [response_summary[u][base_sub_func][context]['change'] for u in units_to_use]
                            presps = [response_summary[u][base_sub_func][context]['prechange'] for u in units_to_use]
                        else:
                            cresps = [response_summary[u][base_sub_func]['active'][context] for u in units_to_use]
                            presps = [response_summary[u][base_sub_func]['passive'][context] for u in units_to_use]
                        # cresps_crossnorm, presps_crossnorm = get_norm_pop_response(units_to_use, state, state, 'change', 'prechange', norm_slice=slice(70,170))
                        # mod_inds = get_norm_diff(cresps_crossnorm, presps_crossnorm, response_slice=slice(70, 170))
                        
                        if metric == 'mod_index':
                            vals = get_mod_index(cresps, presps)
                        elif metric == 'mod_index_norm':
                            vals = get_mod_index_norm(cresps, presps)
                        elif metric == 'auc':
                            vals = np.array([auc_summary[uid][comparison + '_modulation'][context] for uid in units_to_use])
                        elif metric == 'ols':
                            vals = np.array([anova_summary[uid]['model'].params[ols_keys[icontext]] for uid in units_to_use])

                        median_val = np.nanmedian(vals)
                        all_vals.append(vals)
                        # ax.plot(ia, median_val, 'o', color=ccf_utils.get_area_color(area, structure_tree))
                        ci = bootstrap_ci(vals, np.median, num_iterations=num_iterations)
                        error = [[median_val-ci[0]], [ci[1]-median_val]]
                        ax.plot(counter, median_val, 'o', color=color)
                        ax.errorbar([counter], [median_val], yerr=error, color=color)
      

                        counter += 1
                        layer_str = '_' + layer if layer != 'all' else ''
                        cell_type_str = '_' + cell_type if cell_type != 'all' else ''
                        labels.append(f'{area}{layer_str}{cell_type_str}')
                        vals_to_return[context][metric][labels[-1]]=vals
            if plot_comparison_mat:
                vbn_utils.plot_comparison_matrix(*all_vals, ax=ax_comp, labels=labels, cmap='PiYG', colorbar=True, binarize=True, nan_color='lightgray',)

    [ax.set_xticks(np.arange(len(labels))) for ax in axes]
    [ax.set_xticklabels(labels) for ax in axes]
    [vbn_utils.formatFigure(fig, ax) for ax in axes]
    plt.tight_layout()
    if return_vals:
        return vals_to_return


def make_sessionwise_modulation_figure(areas, layers, cell_types, clusters, metrics=['mod_index', 'mod_index_norm', 'auc', 'ols'],
                            comparison='change', flashes=['change', 'prechange',], states=['active', 'passive'], 
                            plot_comparison_mat=False, return_vals = False, num_iterations=1000, base_sub_func=base_sub,
                            response_summary=response_summary, unit_subsample=units, test_func=vbn_utils.hybrid_permutation_test_2, cell_count_thresh=5):
    fig_width = 10
    fig_height = 3*len(metrics)
    fig = plt.figure(figsize=(fig_width, fig_height))
    if comparison=='change':
        numcols = len(states)
    if comparison=='state':
        numcols = len(flashes)
    
    gs = gridspec.GridSpec(len(metrics), numcols, figure=fig)
    axes = []

    if comparison=='change':
        contexts = states
    elif comparison=='state':
        contexts = flashes

    ols_keys = ['C(flash)[T.1]', 'C(flash)[T.1]:C(state)[T.1]'] if comparison=='change' else ['C(state)[T.1]', 'C(flash)[T.1]:C(state)[T.1]'] 
    vals_to_return = {c: {m:{} for m in metrics} for c in contexts}
    for icontext, context in enumerate(contexts):
        for imetric, metric in enumerate(metrics):
            ax = fig.add_subplot(gs[imetric, icontext])
            axes.append(ax)
            if metric=='ols':
                title_add = ols_keys[icontext]
            else:
                title_add = context
            ax.set_title(f'{metric} {title_add}')

            if plot_comparison_mat:
                fig_comp, ax_comp = plt.subplots()
                # fig_comp.set_size_inches(10, 10)
                fig_comp.suptitle(f'{metric} {context}')

            mods = []
            all_aucs = []
            counter = 0
            labels = []
            all_vals = []
            for ia, area in enumerate(make_iterable(areas)):
                for il, layer in enumerate(make_iterable(layers)):
                    for ic, cell_type in enumerate(make_iterable(cell_types)):
                        color = ccf_utils.get_area_color(area, structure_tree)
                        vals = []
                        for session_id in unit_subsample['ecephys_session_id'].unique():
                            units_to_use = get_unit_ids(unit_subsample, area, cell_types=cell_type, layers=layer, clusters=clusters,
                                            clustering='new', session_id=session_id)
                            
                            if len(units_to_use)<cell_count_thresh:
                                vals.append(np.nan)
                                continue
                            
                            if comparison == 'change':
                                cresps = [response_summary[u][base_sub_func][context]['change'] for u in units_to_use]
                                presps = [response_summary[u][base_sub_func][context]['prechange'] for u in units_to_use]
                            else:
                                cresps = [response_summary[u][base_sub_func]['active'][context] for u in units_to_use]
                                presps = [response_summary[u][base_sub_func]['passive'][context] for u in units_to_use]
                            # cresps_crossnorm, presps_crossnorm = get_norm_pop_response(units_to_use, state, state, 'change', 'prechange', norm_slice=slice(70,170))
                            # mod_inds = get_norm_diff(cresps_crossnorm, presps_crossnorm, response_slice=slice(70, 170))
                            
                            if metric == 'mod_index':
                                session_vals = get_mod_index(cresps, presps)
                            elif metric == 'mod_index_norm':
                                session_vals = get_mod_index_norm_njit(np.array(cresps), np.array(presps))
                            elif metric == 'auc':
                                session_vals = np.array([auc_summary[uid][comparison + '_modulation'][context] for uid in units_to_use])
                            elif metric == 'ols':
                                session_vals = np.array([anova_summary[uid]['model'].params[ols_keys[icontext]] for uid in units_to_use])

                            vals.append(np.nanmedian(session_vals))

                        median_val = np.nanmedian(vals)
                        all_vals.append(vals)
                        # ax.plot(ia, median_val, 'o', color=ccf_utils.get_area_color(area, structure_tree))
                        ci = vbn_utils.bootstrap_ci_vectorized(vals, np.median, num_iterations=num_iterations)
                        error = [[median_val-ci[0]], [ci[1]-median_val]]
                        ax.plot(counter, median_val, 'o', color=color)
                        ax.errorbar([counter], [median_val], yerr=error, color=color)
      

                        counter += 1
                        layer_str = '_' + layer if layer != 'all' else ''
                        cell_type_str = '_' + cell_type if cell_type != 'all' else ''
                        labels.append(f'{area}{layer_str}{cell_type_str}')
                        vals_to_return[context][metric][labels[-1]]=vals
            if plot_comparison_mat:
                vbn_utils.plot_comparison_matrix(*all_vals, ax=ax_comp, labels=labels, cmap='PiYG', colorbar=True, binarize=True, test_func=test_func)

    [ax.set_xticks(np.arange(len(labels))) for ax in axes]
    [ax.set_xticklabels(labels) for ax in axes]
    [vbn_utils.formatFigure(fig, ax) for ax in axes]
    plt.tight_layout()
    if return_vals:
        return vals_to_return



In [ ]:
def make_nov_mod_psth_figure(areas, layers, cell_types, clusters,
                            flashes=['prechange', 'change'], state='active', 
                            resp_slice=slice(50,170), display_slice=slice(50, 170), norm=False,
                            plot_pvalues=False, smoothing_kernel_width=3, units_subsample=units):
    
    total_splits = np.prod([len(make_iterable(split)) for split in [areas, layers, cell_types]])
    fig = plt.figure(figsize=(8, 4*total_splits))
    numcols = len(flashes)
    
    gs = gridspec.GridSpec(total_splits, numcols, figure=fig)
    axes = []

    colors = ['b', 'r']
    alphas = {'active': 1, 'passive': 0.5}
    for istate, state_cond in enumerate(make_iterable(state)):
        for iflash, flash in enumerate(flashes):
            
            axcol = iflash
            counter = 0
            labels = []
            for ia, area in enumerate(make_iterable(areas)):
                for il, layer in enumerate(make_iterable(layers)):
                    for ic, cell_type in enumerate(make_iterable(cell_types)):
                        
                        axrow = counter
                        if istate==0:
                            ax = fig.add_subplot(gs[axrow, axcol])
                            axes.append(ax)
                        else:
                            axind = counter + total_splits*iflash
                            ax = axes[axind]
                        ns = []
                        exp_data = []
                        for iexp, experience in enumerate(['Familiar', 'Novel']):
                            units_to_use = get_unit_ids(units_subsample, area, layers=layer, cell_types=cell_type, clusters=clusters, experience=experience)

                            if norm:
                                resps_norm, _ = get_norm_pop_response(units_to_use, state, state, flash, flash, norm_slice=resp_slice)
                            else:
                                resps_norm = base_sub(np.array([flash_data[u][state_cond][flash] for u in units_to_use]))
                                if smoothing_kernel_width>0:
                                    resps_norm = np.array([exponential_convolve(r, smoothing_kernel_width, symmetrical=True) for r in resps_norm])


                            vbn_utils.mean_sem_plot(resps_norm[:, display_slice]*1000, ax, color=colors[iexp], 
                                                    alpha=alphas[state_cond], label=len(resps_norm))
                            exp_data.append(resps_norm[:, display_slice])
                            ns.append(len(resps_norm))
                        
                        if plot_pvalues:
                            ps = [scipy.stats.ranksums(exp_data[0][:, t], exp_data[1][:,t])[1] for t in range(exp_data[0].shape[1])]
                            sigs, corrected_ps = vbn_utils.multiple_comparisons(ps)
                            ax2 = ax.twinx()
                            ax2.plot(corrected_ps<0.01)

                        # if istate==0:
                            # ax.axvspan(20, 100, color='k', alpha=0.2)
                        counter += 1
                        layer_str = '_' + layer if layer != 'all' else ''
                        cell_type_str = '_' + cell_type if cell_type != 'all' else ''
                        labels.append(f'{area}{layer_str}{cell_type_str}')
                        ax.set_title(f'{area}{layer_str}{cell_type_str} \n {flash}')
                        vbn_utils.formatFigure(fig, ax)
                        ax.tick_params(axis='both', labelsize=18)

                        ax.spines['left'].set_bounds(0, 5)
                        ax.set_yticks([0,5])
                        ax.spines['bottom'].set_bounds(20, 100)
                        ax.set_xticks([20,100])

                        ax.legend(frameon=False, handlelength=1, handletextpad=0.2, fontsize=16)
                        ax.set_xlabel('Time from stim start (ms)')
                        ax.set_ylabel('Firing rate (Hz)')

                        for spine in ax.spines.values():
                            spine.set_linewidth(1.5)
                        ax.tick_params(width=1.5)

    plt.tight_layout()


def make_nov_modulation_figure(areas, layers, cell_types, clusters, return_vals=False, 
                            flashes=['prechange', 'change'], units_subsample=units, cell_type_color_dict={}, num_iterations=1000):
    
    fig, axes = plt.subplots(2,2)
    fig.set_size_inches([10,8])
    mods = {s:{f:{} for f in flashes} for s in ['active', 'passive']}
    f_n_resps = {state:{flash:{} for flash in flashes} for state in ['active', 'passive']}

    layer_alphas = [1, 0.6, 0.4, 0.3]

    for istate, state in enumerate(['active', 'passive']):
        for iflash, flash in enumerate(flashes):
            ax = axes[istate][iflash]
            ax.set_title(f'{state} {flash}')

            labels = []
            counter = 0
            for ia, area in enumerate(make_iterable(areas)):
                for il, layer in enumerate(make_iterable(layers)):
                    for ic, cell_type in enumerate(make_iterable(cell_types)):
                        if isinstance(area, list):
                            color = ccf_utils.get_area_color(area[0], structure_tree)
                        else:
                            color = ccf_utils.get_area_color(area, structure_tree)
                        color = cell_type_color_dict.get(cell_type, color)

                        fresps = []
                        nresps = []
                        for resps, exp in zip([fresps, nresps], ['Familiar', 'Novel']):
                            units_to_use = get_unit_ids(units_subsample, area, layers=layer, cell_types=cell_type, clusters=clusters, experience=exp, responsive=False)
                            resps.extend([response_summary[u][base_sub][state][flash] for u in units_to_use])
                        

                        mod_inds = get_nov_mod_index_norm_bootstrap(fresps, nresps, iterations=num_iterations, aggfunc=np.nanmean)
                        nmi = get_nov_mod_index_norm(fresps, nresps, aggfunc=np.nanmean)
                        
                        # ax.plot(ia, nmi, 'o', color=ccf_utils.get_area_color(area, structure_tree))
                        # ci = bootstrap_ci(mod_inds, np.median)
                        # ci = np.percentile(mod_inds, [2.5, 97.5])
                        # error = [[nmi-ci[0]], [ci[1]-nmi]]
                        error = np.nanstd(mod_inds)
                        ax.plot(counter, nmi, 'o', color=color, alpha=layer_alphas[il], ms=10)
                        ax.errorbar([counter], [nmi], yerr=error, color=color, alpha=layer_alphas[il])
                        
                        layer_str = '_' + layer if layer != 'all' else ''
                        cell_type_str = '_' + cell_type if cell_type != 'all' else ''
                        labels.append(f'{area}{layer_str}{cell_type_str}')
                        mods[state][flash][labels[-1]] = mod_inds
                        f_n_resps[state][flash][labels[-1]] = [fresps, nresps]
                        counter += 1

            ax.set_xticks(np.arange(len(labels)))
            ax.set_xticklabels(labels, rotation=90)
            vbn_utils.formatFigure(fig, ax, yLabel='Novelty Modulation')
            plt.tight_layout()
    if return_vals:
        return mods, f_n_resps

# run novelty modulation for each mouse separately 
session_table = pd.read_csv(sessions_table_file)
sessions_no_abnorm = sessions_table[sessions_table['abnormal_activity'].isnull() & sessions_table['abnormal_histology'].isnull()]
mice_with_fam_and_nov_sessions = np.intersect1d(sessions_no_abnorm[sessions_no_abnorm['experience_level']=='Familiar']['mouse_id'].values, sessions_no_abnorm[sessions_no_abnorm['experience_level']=='Novel']['mouse_id'].values)
def make_mousewise_nov_modulation_figure(areas, layers, cell_types, clusters, return_vals=False, 
                            flashes=['prechange', 'change'], units_subsample=units, cell_type_color_dict={}, num_iterations=1000, cell_count_thresh=5):
    
    fig, axes = plt.subplots(2,2)
    fig.set_size_inches([12,8])
    mods = {s:{f:{} for f in flashes} for s in ['active', 'passive']}

    layer_alphas = [1, 0.6, 0.4, 0.3]

    for istate, state in enumerate(['active', 'passive']):
        for iflash, flash in enumerate(flashes):
            ax = axes[istate][iflash]
            ax.set_title(f'{state} {flash}')

            labels = []
            counter = 0
            for ia, area in enumerate(make_iterable(areas)):
                for il, layer in enumerate(make_iterable(layers)):
                    for ic, cell_type in enumerate(make_iterable(cell_types)):
                        if isinstance(area, list):
                            color = ccf_utils.get_area_color(area[0], structure_tree)
                        else:
                            color = ccf_utils.get_area_color(area, structure_tree)
                        color = cell_type_color_dict.get(cell_type, color)
                        mod_inds = []
                        for imouse, mouse in enumerate(mice_with_fam_and_nov_sessions):
                            fresps = []
                            nresps = []
                            for resps, exp in zip([fresps, nresps], ['Familiar', 'Novel']):
                                mouse_session = sessions_table[(sessions_table['mouse_id']==mouse)&(sessions_table['experience_level']==exp)]['ecephys_session_id'].values[0]
                                units_to_use = get_unit_ids(units_subsample, area, layers=layer, cell_types=cell_type, clusters=clusters, experience=exp, responsive=False, session_id=mouse_session)
                                resps.extend([response_summary[u][base_sub][state][flash] for u in units_to_use])
                            
                            if len(fresps)>cell_count_thresh and len(nresps)>cell_count_thresh:
                                # mouse_mod_inds = get_nov_mod_index_norm(fresps, nresps, iterations=num_iterations, aggfunc=np.nanmean)
                                # mod_inds.append(np.nanmedian(mouse_mod_inds))
                                mouse_mod_ind = vbn_utils.norm_novel_modulation_ind(fresps, nresps)
                                mod_inds.append(mouse_mod_ind)
                            else:
                                mod_inds.append(np.nan)
                        
                        median_val = np.nanmedian(mod_inds)
                        # ax.plot(ia, median_val, 'o', color=ccf_utils.get_area_color(area, structure_tree))
                        ci = bootstrap_ci(mod_inds, np.nanmedian)
                        # ci = np.percentile(mod_inds, [2.5, 97.5])
                        error = [[median_val-ci[0]], [ci[1]-median_val]]
                        ax.plot(counter, median_val, 'o', color=color, alpha=layer_alphas[il], ms=10)
                        ax.errorbar([counter], [median_val], yerr=error, color=color, alpha=layer_alphas[il])
                        
                        layer_str = '_' + layer if layer != 'all' else ''
                        cell_type_str = '_' + cell_type if cell_type != 'all' else ''
                        labels.append(f'{area}{layer_str}{cell_type_str}')
                        mods[state][flash][labels[-1]] = mod_inds

                        counter += 1

            ax.set_xticks(np.arange(len(labels)))
            ax.set_xticklabels(labels, rotation=90)
            vbn_utils.formatFigure(fig, ax, yLabel='Novelty Modulation')
            plt.tight_layout()
    if return_vals:
        return mods


def make_novelty_vs_context_figure(areas, layers, cell_types, clusters,
                            flashes = ['nonshared_nonchange', 'shared_nonchange'], state='active', 
                            resp_slice=slice(50,170), display_slice=slice(50, 170), norm=False,
                            plot_pvalues=False, smoothing_kernel_width=3, units_subsample=units,):
    
    total_splits = np.prod([len(make_iterable(split)) for split in [areas, layers, cell_types]])
    fig = plt.figure(figsize=(5, 4*total_splits))
    numcols = 1
    
    gs = gridspec.GridSpec(total_splits, numcols, figure=fig)
    axes = []

    color_dict = {'nonshared_Familiar': 'b',
                  'nonshared_Novel': 'r',
                  'shared_Familiar': 'g',
                  'shared_Novel': 'purple'}
    
    alphas = {'active': 1, 'passive': 0.5}
    for istate, state_cond in enumerate(make_iterable(state)):
        for iflash, flash in enumerate(flashes):
            
            axcol = 0
            counter = 0
            labels = []
            for ia, area in enumerate(make_iterable(areas)):
                for il, layer in enumerate(make_iterable(layers)):
                    for ic, cell_type in enumerate(make_iterable(cell_types)):
                        
                        axrow = counter
                        if iflash==0:
                            ax = fig.add_subplot(gs[axrow, axcol])
                            axes.append(ax)
                        else:
                            axind = counter + total_splits*iflash
                            ax = axes[counter]
                        ns = []
                        exp_data = []
                        for iexp, experience in enumerate(['Familiar', 'Novel']):
                            units_to_use = get_unit_ids(units_subsample, area, layers=layer, cell_types=cell_type, clusters=clusters, experience=experience)

                            if norm:
                                resps_norm, _ = get_norm_pop_response(units_to_use, state, state, flash, flash, norm_slice=resp_slice)
                            else:
                                resps_norm = base_sub(np.array([flash_data[u][state_cond][flash] for u in units_to_use]))
                                if smoothing_kernel_width>0:
                                    resps_norm = np.array([exponential_convolve(r, smoothing_kernel_width, symmetrical=True) for r in resps_norm])


                            vbn_utils.mean_sem_plot(resps_norm[:, display_slice]*1000, ax, color=color_dict[f'{flash.split("_")[0]}_{experience}'], 
                                                    alpha=alphas[state_cond], label=len(resps_norm))
                            exp_data.append(resps_norm[:, display_slice])
                            ns.append(len(resps_norm))
                        
                        if plot_pvalues:
                            ps = [scipy.stats.ranksums(exp_data[0][:, t], exp_data[1][:,t])[1] for t in range(exp_data[0].shape[1])]
                            sigs, corrected_ps = vbn_utils.multiple_comparisons(ps)
                            ax2 = ax.twinx()
                            ax2.plot(corrected_ps<0.01)

                        # if istate==0:
                            # ax.axvspan(20, 100, color='k', alpha=0.2)
                        counter += 1
                        layer_str = '_' + layer if layer != 'all' else ''
                        cell_type_str = '_' + cell_type if cell_type != 'all' else ''
                        labels.append(f'{area}{layer_str}{cell_type_str}')
                        ax.set_title(f'{area}{layer_str}{cell_type_str}')
                        vbn_utils.formatFigure(fig, ax)

                        ax.spines['left'].set_bounds(0, 5)
                        ax.set_yticks([0,5])
                        ax.spines['bottom'].set_bounds(20, 100)
                        ax.set_xticks([20,100])

                        ax.legend(frameon=False, fontsize=16, handlelength=1, handletextpad=0.2)
                        ax.set_xlabel('Time from stim start (ms)')
                        ax.set_ylabel('Firing rate (Hz)')
                        ax.tick_params(axis='both', labelsize=20)

    plt.tight_layout()

## Change/State modulation across areas

In [ ]:
import warnings
warnings.filterwarnings('ignore')

areas = ['VISp', 'VISl', 'VISal', 'VISrl', 'VISpm', 'VISam', 'VISall', 'LGd', 'LP', 'Hipp', 'SCMRN']
layers = ['2/3', '4', '5', '6', 'all']

base_slice = slice(0, 50)
resp_slice = slice(70, 150)
binsize = 5
time = np.arange(-50, 750, binsize)

base_funcs = [base_sub, no_base_sub]

unitIDs = units.loc[du.apply_unit_quality_filter(units)]['unit_id'].values

response_summary = {u:{base_func: {cond: {flash: [] for flash in ['change', 'prechange','shared_nonchange', 'nonshared_nonchange']} for cond in ['active', 'passive']} for base_func in base_funcs} for u in unitIDs}

for base_func in base_funcs:
    # for cond, linestyle in zip(['active', 'passive'], ['solid', 'dotted']):
    for cond in ['active', 'passive']:
        for flash in ['change', 'prechange','shared_nonchange', 'nonshared_nonchange']:
            cdata = base_func(np.stack([flash_data[u][cond][flash] for u in unitIDs])*1000)
            cresponses = np.mean(cdata[:, resp_slice], axis=1)

            for uid, u in enumerate(unitIDs):
                response_summary[u][base_func][cond][flash] = cresponses[uid]


In [ ]:
plt.rcParams['font.size'] = 18
areas_ordered = ['LGd', 'LP', 'VISp', 'VISl', 'VISrl', 'VISal', 'VISpm', 'VISam', 'Hipp', 'SCMRN']
make_psth_figure(areas_ordered, layers='all', cell_types='all', clusters='sensory', comparison='change', unit_subsample=units,
                 norm=False, split=False, use_ccf_color=False, smooth_kernel_width=3, base_sub_func=base_sub)

In [ ]:
areas_ordered = ['LGd', 'LP', 'VISp', 'VISl', 'VISrl', 'VISal', 'VISpm', 'VISam', 'SCMRN']

change_vals = make_modulation_figure(areas_ordered, layers='all', cell_types='all', clusters='sensory', comparison='change', 
                metrics=['mod_index_norm',], plot_comparison_mat=True, return_vals=True, num_iterations=10, base_sub_func=base_sub)
state_vals = make_modulation_figure(areas_ordered, layers='all', cell_types='all', clusters='sensory', comparison='state', 
                metrics=['mod_index_norm',], plot_comparison_mat=True, return_vals=True, num_iterations=10, base_sub_func=base_sub)

fig, ax = plt.subplots()
for ia, area in enumerate(areas_ordered):
    color = ccf_utils.get_area_color(area, structure_tree)
    fcs = [color, 'w']
    for ivals, vals in enumerate([change_vals['active']['mod_index_norm'][area], state_vals['change']['mod_index_norm'][area]]):

        median_val = np.nanmedian(vals)
        # ax.plot(ia, median_val, 'o', color=ccf_utils.get_area_color(area, structure_tree))
        ci = bootstrap_ci(vals, np.median, num_iterations=1000)
        error = [[median_val-ci[0]], [ci[1]-median_val]]
        ax.plot(ia, median_val, 'o', mec=color, mfc=fcs[ivals], ms=12, markeredgewidth=2)
        ax.errorbar([ia], [median_val], yerr=error, color=color)

ax.axhline(0, ls='dotted', color='k')
ax.set_xticks(np.arange(len(areas_ordered)))
ax.set_xticklabels(areas_ordered[:-1] + ['SCm/MRN'], rotation=45,)
ax.tick_params(axis='both', labelsize=20)  # Change font size for both x and y axes
ax.set_ylabel('Modulation Index', fontsize=20)
for spine in ax.spines.values():
    spine.set_linewidth(1.5)  # Set spine line width to 2 points

# Increase the line width of ticks
ax.tick_params(width=1.5)  # Set tick line width to 2 points

vbn_utils.formatFigure(fig, ax)

### Sessionwise variability in response

In [ ]:
areas_ordered = ['LGd', 'LP', 'VISp', 'VISl', 'VISrl', 'VISal', 'VISpm', 'VISam', 'SCMRN']

change_vals = make_sessionwise_modulation_figure(areas_ordered, layers='all', cell_types='all', clusters='sensory', comparison='change', 
                metrics=['mod_index_norm',], plot_comparison_mat=True, return_vals=True, num_iterations=1, base_sub_func=base_sub, test_func=vbn_utils.hybrid_permutation_test_2, cell_count_thresh=5)
state_vals = make_sessionwise_modulation_figure(areas_ordered, layers='all', cell_types='all', clusters='sensory', comparison='state', 
                metrics=['mod_index_norm',], plot_comparison_mat=True, return_vals=True, num_iterations=1, base_sub_func=base_sub, test_func=vbn_utils.hybrid_permutation_test_2, cell_count_thresh=5)

fig, ax = plt.subplots()
for ia, area in enumerate(areas_ordered):
    color = ccf_utils.get_area_color(area, structure_tree)
    fcs = [color, 'w']
    for ivals, vals in enumerate([change_vals['active']['mod_index_norm'][area], state_vals['change']['mod_index_norm'][area]]):
        vals = np.array(vals)
        median_val = np.nanmedian(vals)
        # ax.plot(ia, median_val, 'o', color=ccf_utils.get_area_color(area, structure_tree))
        ci = vbn_utils.bootstrap_ci_vectorized(vals, np.median, num_iterations=1000)
        error = [[median_val-ci[0]], [ci[1]-median_val]]
        # error = np.percentile(vals[~np.isnan(vals)], [2.5, 97.5])
        # error = [[median_val-error[0]], [error[1]-median_val]]
        # error = np.nanstd(vals)
        ax.plot(ia, median_val, 'o', mec=color, mfc=fcs[ivals], ms=12, markeredgewidth=2)
        ax.errorbar([ia], [median_val], yerr=error, color=color)

ax.axhline(0, ls='dotted', color='k')
ax.set_xticks(np.arange(len(areas_ordered)))
ax.set_xticklabels(areas_ordered[:-1] + ['SCm/MRN'], rotation=45,)
ax.tick_params(axis='both', labelsize=20)  # Change font size for both x and y axes
ax.set_ylabel('Modulation Index', fontsize=20)
for spine in ax.spines.values():
    spine.set_linewidth(1.5)  # Set spine line width to 2 points

# Increase the line width of ticks
ax.tick_params(width=1.5)  # Set tick line width to 2 points

vbn_utils.formatFigure(fig, ax)

for ia, area in enumerate(areas_ordered):
    change_counts = np.sum(~np.isnan(change_vals['active']['mod_index_norm'][area]))
    ax.text(ia, ax.get_ylim()[1], f'{change_counts}', ha='center', va='center', fontsize=20)


area_sessionwise_change_vals = change_vals
area_sessionwise_state_vals = state_vals

In [ ]:
for vals, label in zip([area_sessionwise_change_vals['active']['mod_index_norm'], area_sessionwise_state_vals['change']['mod_index_norm']], ['Change Modulation', 'State Modulation']):
    fig, ax = plt.subplots()
    fig.suptitle(label)
    vbn_utils.plot_comparison_matrix(*vals.values(), ax=ax, labels=list(vals.keys()), test_func=vbn_utils.hybrid_permutation_test_2, cmap='PiYG', colorbar=True, binarize=True, nan_color='lightgray')

In [ ]:
for area in areas_ordered:
    fig, ax = plt.subplots()
    mean_ch_vals = np.nanmean([change_vals[context]['mod_index_norm'][area] for context in ['active', 'passive']], axis=0)
    mean_state_vals = np.nanmean([state_vals[context]['mod_index_norm'][area] for context in ['prechange', 'change']], axis=0)
    ax.plot(mean_ch_vals, mean_state_vals, 'k.')
    ax.set_xlabel('change mod')
    ax.set_ylabel('state mod')

    mask = ~np.isnan(mean_ch_vals) & ~np.isnan(mean_state_vals) & ~np.isinf(mean_ch_vals) & ~np.isinf(mean_state_vals)
    mean_ch_vals_clean = mean_ch_vals[mask]
    mean_state_vals_clean = mean_state_vals[mask]
    slope, intercept, r_value, p_value, std_err = scipy.stats.linregress(mean_ch_vals_clean, mean_state_vals_clean)
    fig.suptitle(f'{area} r={r_value:.2f} p={p_value:.3f}')

In [ ]:
# save change_vals and state_vals to pickle
import pickle
with open("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/change_modulation_across_areas.pkl", 'wb') as file:
    pickle.dump(change_vals, file)

with open("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/state_modulation_across_areas.pkl", 'wb') as file:
    pickle.dump(state_vals, file)

In [ ]:
change_vals = make_modulation_figure(areas_ordered, layers='all', cell_types='all', clusters='sensory', comparison='change', 
                metrics=['mod_index_norm',], plot_comparison_mat=True, return_vals=True)

In [ ]:
make_psth_figure(areas_ordered, layers='all', cell_types='all', clusters='sensory', comparison='state', norm=False)

In [ ]:
make_modulation_figure(areas_ordered, layers='all', cell_types='all', clusters='sensory', comparison='state')

In [ ]:
state_vals = make_modulation_figure(areas_ordered, layers='all', cell_types='all', clusters='sensory', comparison='state', 
                metrics=['mod_index_norm',], plot_comparison_mat=True, return_vals=True)

## Change/State modulation across layers

In [ ]:
make_psth_figure('VISall', layers=['2/3', '4', '5', '6'], cell_types='all', clusters='sensory', 
                comparison='change', norm=False, split=False, use_ccf_color=False,
                unit_subsample=units, base_sub_func=no_base_sub, )


In [ ]:
layers=['2/3', '4', '5', '6']

regions = ['VISall', 'VISp', 'VISl', 'VISrl', 'VISal', 'VISpm', 'VISam']

change_vals = make_modulation_figure(regions, layers=layers, cell_types='all', clusters='sensory', comparison='change', 
                metrics=['mod_index_norm',], plot_comparison_mat=True, return_vals=True, num_iterations=10)
state_vals = make_modulation_figure(regions, layers=layers, cell_types='all', clusters='sensory', comparison='state', 
                metrics=['mod_index_norm',], plot_comparison_mat=True, return_vals=True, num_iterations=10)

In [ ]:
# save change_vals and state_vals to pickle
import pickle
with open("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/change_modulation_across_layers.pkl", 'wb') as file:
    pickle.dump(change_vals, file)
with open("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/state_modulation_across_layers.pkl", 'wb') as file:
    pickle.dump(state_vals, file)

In [ ]:
layers=['2/3', '4', '5', '6']

change_vals = make_modulation_figure('VISall', layers=layers, cell_types='all', clusters='sensory', comparison='change', 
                metrics=['mod_index_norm',], plot_comparison_mat=True, return_vals=True, num_iterations=10)
state_vals = make_modulation_figure('VISall', layers=layers, cell_types='all', clusters='sensory', comparison='state', 
                metrics=['mod_index_norm',], plot_comparison_mat=True, return_vals=True, num_iterations=10)

fig, ax = plt.subplots()
alphas = [1, 0.75, 0.5, 0.35]
for il, layer in enumerate(layers):
    color = ccf_utils.get_area_color('VISall', structure_tree)
    fcs = [color, 'w']

    for ivals, vals in enumerate([change_vals['active']['mod_index_norm']['VISall_'+layer], 
                                    state_vals['change']['mod_index_norm']['VISall_'+layer]]):

        median_val = np.nanmedian(vals)
        # ax.plot(ia, median_val, 'o', color=ccf_utils.get_area_color(area, structure_tree))
        ci = bootstrap_ci(vals, np.median, num_iterations=1000)
        error = [[median_val-ci[0]], [ci[1]-median_val]]
        ax.plot(il, median_val, 'o', mec=color, mfc=fcs[ivals], ms=12, alpha=alphas[il], markeredgewidth=3)
        ax.errorbar([il], [median_val], yerr=error, color=color, alpha=alphas[il])

ax.axhline(0, ls='dotted', color='k')
ax.set_xticks(np.arange(len(layers)))
ax.set_xticklabels(layers,)
ax.tick_params(axis='both', labelsize=20)  # Change font size for both x and y axes
ax.set_ylabel('Modulation Index', fontsize=20)
for spine in ax.spines.values():
    spine.set_linewidth(1.5)  # Set spine line width to 2 points

# Increase the line width of ticks
ax.tick_params(width=1.5)  # Set tick line width to 2 points
vbn_utils.formatFigure(fig, ax)

In [ ]:
plt.rcParams['font.size'] = 26
for vals, label in zip([layer_change_vals['active']['mod_index_norm'], layer_state_vals['change']['mod_index_norm']], ['Change Modulation', 'State Modulation']):
    
    fig, ax = plt.subplots()
    fig.suptitle(label)
    vbn_utils.plot_comparison_matrix(*vals.values(), ax=ax, labels=list(vals.keys()), test_func=vbn_utils.hybrid_permutation_test_2, cmap='PiYG', colorbar=True, binarize=True, nan_color='lightgray')

In [ ]:
make_modulation_figure('VISall', layers=['2/3', '4', '5', '6'], cell_types='all', clusters='sensory', comparison='change')

In [ ]:
make_modulation_figure('VISall', layers=['2/3', '4', '5', '6'], cell_types='all', clusters='sensory', 
                    comparison='change', metrics=['mod_index_norm',], plot_comparison_mat=True)

In [ ]:
make_psth_figure('VISall', layers=['2/3', '4', '5', '6'], cell_types='all', clusters='sensory', comparison='state', norm=False)

In [ ]:
make_modulation_figure('VISall', layers=['2/3', '4', '5', '6'], cell_types='all', clusters='sensory', comparison='state')

In [ ]:
make_modulation_figure('VISall', layers=['2/3', '4', '5', '6'], cell_types='all', clusters='sensory', 
                    comparison='state', metrics=['mod_index_norm',], plot_comparison_mat=True)

## Change/State modulation across cell types

In [ ]:
make_psth_figure('VISall', layers='all', cell_types=['RS', 'FS', 'SST', 'VIP'], clusters='sensory', 
                comparison='change', norm=False, split=False, use_ccf_color=False, smooth_kernel_width=3,
                base_sub_func=no_base_sub, unit_subsample=low_baseline_drift_units)

In [ ]:
cell_types=['RS', 'FS', 'SST', 'VIP']
cell_type_colors = {'RS': 'teal', 
                    'FS': 'red',
                    'SST': 'dodgerblue',
                    'VIP': 'mediumorchid'}

regions = ['VISall', 'VISp', 'VISl', 'VISrl', 'VISal', 'VISpm', 'VISam']

change_vals = make_modulation_figure(regions, layers='all', cell_types=cell_types, clusters='sensory', comparison='change', 
                metrics=['mod_index_norm',], plot_comparison_mat=True, return_vals=True, num_iterations=10)
state_vals = make_modulation_figure(regions, layers='all', cell_types=cell_types, clusters='sensory', comparison='state', 
                metrics=['mod_index_norm',], plot_comparison_mat=True, return_vals=True, num_iterations=10)

In [ ]:
# save change_vals and state_vals to pickle
import pickle
with open("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/change_modulation_across_celltypes.pkl", 'wb') as file:
    pickle.dump(change_vals, file)
with open("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/state_modulation_across_celltypes.pkl", 'wb') as file:
    pickle.dump(state_vals, file)

In [ ]:
cell_types=['RS', 'FS', 'SST', 'VIP']
cell_type_colors = {'RS': 'teal', 
                    'FS': 'red',
                    'SST': 'dodgerblue',
                    'VIP': 'mediumorchid'}

change_vals = make_modulation_figure('VISall', layers='all', cell_types=cell_types, clusters='sensory', comparison='change', 
                metrics=['mod_index_norm',], plot_comparison_mat=True, return_vals=True, num_iterations=10)
state_vals = make_modulation_figure('VISall', layers='all', cell_types=cell_types, clusters='sensory', comparison='state', 
                metrics=['mod_index_norm',], plot_comparison_mat=True, return_vals=True, num_iterations=10)

fig, ax = plt.subplots()
alphas = [1, 0.75, 0.5, 0.35]
for ic, cell_type in enumerate(cell_types):
    color = cell_type_colors[cell_type]
    fcs = [color, 'w']

    for ivals, vals in enumerate([change_vals['active']['mod_index_norm']['VISall_'+cell_type], 
                                    state_vals['change']['mod_index_norm']['VISall_'+cell_type]]):

        median_val = np.nanmedian(vals)
        # ax.plot(ia, median_val, 'o', color=ccf_utils.get_area_color(area, structure_tree))
        ci = bootstrap_ci(vals, np.median, num_iterations=1000)
        error = [[median_val-ci[0]], [ci[1]-median_val]]
        ax.plot(ic, median_val, 'o', mec=color, mfc=fcs[ivals], ms=12, markeredgewidth=2)
        ax.errorbar([ic], [median_val], yerr=error, color=color)
ax.axhline(0, ls='dotted', color='k')
ax.set_xticks(np.arange(len(cell_types)))
ax.set_xticklabels(cell_types,)
ax.tick_params(axis='both', labelsize=20)  # Change font size for both x and y axes
ax.set_ylabel('Modulation Index', fontsize=20)
for spine in ax.spines.values():
    spine.set_linewidth(1.5)  # Set spine line width to 2 points

# Increase the line width of ticks
ax.tick_params(width=1.5)  # Set tick line width to 2 points
vbn_utils.formatFigure(fig, ax)

### Sessionwise

In [ ]:
cell_types=['RS', 'FS', 'SST',]# 'VIP']
cell_type_colors = {'RS': 'teal', 
                    'FS': 'red',
                    'SST': 'dodgerblue',
                    'VIP': 'mediumorchid'}

change_vals = make_sessionwise_modulation_figure('VISall', layers='all', cell_types=cell_types, clusters='sensory', comparison='change', 
                metrics=['mod_index_norm',], plot_comparison_mat=True, return_vals=True, num_iterations=1, test_func=vbn_utils.hybrid_permutation_test_2, cell_count_thresh=5)
state_vals = make_sessionwise_modulation_figure('VISall', layers='all', cell_types=cell_types, clusters='sensory', comparison='state', 
                metrics=['mod_index_norm',], plot_comparison_mat=True, return_vals=True, num_iterations=1, test_func=vbn_utils.hybrid_permutation_test_2, cell_count_thresh=5)

fig, ax = plt.subplots()
alphas = [1, 0.75, 0.5, 0.35]
for ic, cell_type in enumerate(cell_types):
    color = cell_type_colors[cell_type]
    fcs = [color, 'w']

    for ivals, vals in enumerate([change_vals['active']['mod_index_norm']['VISall_'+cell_type], 
                                    state_vals['change']['mod_index_norm']['VISall_'+cell_type]]):

        median_val = np.nanmedian(vals)
        # ax.plot(ia, median_val, 'o', color=ccf_utils.get_area_color(area, structure_tree))
        ci = vbn_utils.bootstrap_ci_vectorized(vals, np.median, num_iterations=1000)
        error = [[median_val-ci[0]], [ci[1]-median_val]]
        ax.plot(ic, median_val, 'o', mec=color, mfc=fcs[ivals], ms=12, markeredgewidth=2)
        ax.errorbar([ic], [median_val], yerr=error, color=color)
ax.axhline(0, ls='dotted', color='k')
ax.set_xticks(np.arange(len(cell_types)))
ax.set_xticklabels(cell_types, rotation=45)
ax.tick_params(axis='both', labelsize=20)  # Change font size for both x and y axes
ax.set_ylabel('Modulation Index', fontsize=20)
for spine in ax.spines.values():
    spine.set_linewidth(1.5)  # Set spine line width to 2 points

# Increase the line width of ticks
ax.tick_params(width=1.5)  # Set tick line width to 2 points
vbn_utils.formatFigure(fig, ax)

for ic, cell_type in enumerate(cell_types):
    change_counts = np.sum(~np.isnan(change_vals['active']['mod_index_norm']['VISall_'+cell_type]))
    ax.text(ic, ax.get_ylim()[1], f'{change_counts}', ha='center', va='center', fontsize=20)

celltype_change_vals = change_vals
celltype_state_vals = state_vals

In [ ]:
plt.rcParams['font.size'] = 26

for vals, label in zip([celltype_change_vals['active']['mod_index_norm'], celltype_state_vals['change']['mod_index_norm']], ['Change Modulation', 'State Modulation']):
    
    fig, ax = plt.subplots()
    fig.suptitle(label)
    vbn_utils.plot_comparison_matrix(*vals.values(), ax=ax, labels=list(vals.keys()), test_func=vbn_utils.hybrid_permutation_test_2, cmap='PiYG', colorbar=True, binarize=True, nan_color='lightgray')

In [ ]:
make_modulation_figure('VISall', layers='all', cell_types=['RS', 'FS', 'SST', 'VIP'], clusters='sensory', comparison='change',
    metrics=['mod_index_norm',], plot_comparison_mat=True)

In [ ]:
make_modulation_figure('VISall', layers='all', cell_types=['RS', 'FS', 'SST', 'VIP'], clusters='sensory', comparison='change')

In [ ]:
make_psth_figure('VISall', layers='all', cell_types=['RS', 'FS', 'SST', 'VIP'], clusters='sensory', comparison='state', norm=False)


In [ ]:
# make_psth_figure('VISall', layers='all', cell_types=['RS', 'FS', 'SST', 'VIP'], clusters='sensory', comparison='state')
make_modulation_figure('VISall', layers='all', cell_types=['RS', 'FS', 'SST', 'VIP'], clusters='sensory', comparison='state')

In [ ]:
make_modulation_figure('VISall', layers='all', cell_types=['RS', 'FS', 'SST', 'VIP'], clusters='sensory', 
                comparison='state', metrics=['mod_index_norm',], plot_comparison_mat=True)

## Response latencies across conditions

In [ ]:
stim_filters = vbn_utils.get_stim_filters()


In [ ]:
fig, ax = plt.subplots()
for ic, cell_type in enumerate(['RS', 'FS']):
    cond_ttfs = []
    for istate, tensor in enumerate([active_tensor_file, passive_tensor_file]):
        unit_ids = vbn_utils.get_unit_ids(units, 'VISall', cell_types=cell_type, layers='all', clusters=1, clustering='new')
        session_list = units.set_index('unit_id').loc[unit_ids]['ecephys_session_id'].unique()
        ttfs, unitIDs = vbn_utils.unit_averaged_psth(tensor, stim_table, session_list, unit_ids, *stim_filters['nonchange_nolick'], baseline_length=50, resp_window_length=750,
                                just_get_baselines=False, time_to_first_spike=True)
        all_ttfs = np.concatenate(ttfs)
        print(all_ttfs.shape)
        all_ttfs = all_ttfs[~np.isnan(all_ttfs)]
        cond_ttfs.append(all_ttfs)
        ax.boxplot(all_ttfs, positions=[ic+0.1*istate], showfliers=False, showmeans=True)
    print(scipy.stats.kstest(*cond_ttfs))


# Novelty Modulation

## Novelty modulation across areas

In [ ]:
areas_ordered = ['LGd', 'LP', 'VISp', 'VISl', 'VISrl', 'VISal', 'VISpm', 'VISam', 'SCMRN']
make_nov_mod_psth_figure(areas_ordered, layers='all', cell_types='all', clusters='all', norm=False, state=['active', 'passive'],
                        flashes=['nonshared_nonchange', 'change'], plot_pvalues=False, smoothing_kernel_width=3, units_subsample=units)
vals = make_nov_modulation_figure(areas_ordered, layers='all', cell_types='all', clusters='all', return_vals=True,
                        flashes=['nonshared_nonchange', 'change'], units_subsample=units, num_iterations=100)

In [ ]:
# save vals to pickle
import pickle
with open("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/novelty_modulation_across_areas.pkl", 'wb') as file:
    pickle.dump(vals, file)

### Mousewise novelty modulation

In [ ]:
# areas_ordered = ['LGd', 'LP', 'VISall', 'VISp', 'VISl', 'VISrl', 'VISal', 'VISpm', 'VISam', 'SCMRN']
areas_ordered = ['LGd', 'LP', 'VISall','SCMRN']
# make_nov_mod_psth_figure(areas_ordered, layers='all', cell_types='all', clusters='all', norm=False, state=['active', 'passive'],
#                         flashes=['nonshared_nonchange', 'change'], plot_pvalues=False, smoothing_kernel_width=3, units_subsample=units)
vals = make_mousewise_nov_modulation_figure(areas_ordered, layers='all', cell_types='all', clusters='all', return_vals=True,
                        flashes=['nonshared_nonchange', 'change'], units_subsample=units, num_iterations=1000, cell_count_thresh=5)

for ia, area in enumerate(areas_ordered):
    area_mouse_count = np.sum(~np.isnan(vals['active']['change'][f'{area}']))
    print(f'{area}: {area_mouse_count} mice')

area_mousewise_noveltymod_vals = vals

In [ ]:
plt.rcParams['font.size'] = 24
for state in ['active', 'passive']:
    for flash in ['nonshared_nonchange', 'change']:
        fig, ax = plt.subplots()
        fig.suptitle(f'{state} {flash}')
        condition_vals = area_mousewise_noveltymod_vals[state][flash]
        vbn_utils.plot_comparison_matrix(*area_mousewise_noveltymod_vals[state][flash].values(), ax=ax,labels=list(area_mousewise_noveltymod_vals[state][flash].keys()), test_func=vbn_utils.hybrid_permutation_test_2, cmap='PiYG',
                                        colorbar=True, binarize=True, nan_color='lightgray')

In [ ]:
make_nov_mod_psth_figure('VISall', layers='all', cell_types='RS', clusters='all', norm=False, state=['active',],
                        flashes=['nonshared_nonchange', 'change'], plot_pvalues=False, smoothing_kernel_width=3, units_subsample=units.set_index('unit_id').loc[ggh].reset_index())
make_nov_mod_psth_figure('VISall', layers='all', cell_types='RS', clusters='all', norm=False, state=['active',],
                        flashes=['nonshared_nonchange', 'change'], plot_pvalues=False, smoothing_kernel_width=3, units_subsample=units.set_index('unit_id').loc[hhg].reset_index())
make_nov_mod_psth_figure('VISall', layers='all', cell_types='RS', clusters='all', norm=False, state=['active',],
                        flashes=['nonshared_nonchange', 'change'], plot_pvalues=False, smoothing_kernel_width=3, units_subsample=units.set_index('unit_id').loc[ghg].reset_index())

## Novelty modulation across layers

In [ ]:
layers = ['2/3', '4', '5', '6']
ctx_areas_ordered = ['VISp', 'VISl', 'VISrl', 'VISal', 'VISpm', 'VISam', 'VISall']
make_nov_mod_psth_figure(ctx_areas_ordered, layers=layers, cell_types='all', clusters='all', norm=False, units_subsample=units, flashes=['nonshared_nonchange', 'change'])
# make_nov_modulation_figure(ctx_areas_ordered, layers=layers, cell_types='all', clusters='all', units_subsample=units)
# vals = make_nov_modulation_figure(ctx_areas_ordered, layers=layers, cell_types='all', clusters='all', units_subsample=h_to_g_units)

In [ ]:
ctx_areas_ordered = ['VISall', 'VISp', 'VISl', 'VISrl', 'VISal', 'VISpm', 'VISam']

vals = make_nov_modulation_figure(ctx_areas_ordered, layers=layers, cell_types='all', clusters='all', units_subsample=units, 
                        flashes=['nonshared_nonchange', 'change'], return_vals=True)

In [ ]:
# save vals to pickle
import pickle
with open("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/novelty_modulation_across_layers.pkl", 'wb') as file:
    pickle.dump(vals, file)


In [ ]:
from matplotlib.colors import TwoSlopeNorm
from matplotlib.colors import Normalize

# Create a divergent colormap centered on 0
plt.rcParams['font.size'] = 16  # Set the font size for the entire plot

val_matrix = np.full((len(ctx_areas_ordered), 4), np.nan)
for ia, area in enumerate(ctx_areas_ordered):
    for il, layer in enumerate(['2/3', '4', '5', '6']):
       key = f'{area}_{layer}'
       val_matrix[ia, il] = np.nanmedian(vals['active']['change'][key])

cmap = plt.cm.bwr  # Use the 'bwr' colormap
# norm = TwoSlopeNorm(vcenter=0)  # Automatically scale with 0 at the center
maxval = np.max(np.abs(val_matrix))
norm = Normalize(vmin=-maxval, vmax=maxval)  # Normalize to the min and max of the matrix

fig, ax = plt.subplots()
im = ax.imshow(val_matrix.T, cmap=cmap, norm=norm, aspect='equal')
ax.set_yticks(np.arange(4))
ax.set_yticklabels(['2/3', '4', '5', '6'])
ax.set_xticks(np.arange(len(ctx_areas_ordered)))
ax.set_xticklabels(ctx_areas_ordered, rotation=90)
ax.set_xlabel('Area')
ax.set_ylabel('Layer')
# vbn_utils.formatFigure(fig, ax)
# plt.colorbar(im, label='Novelty Modulation Index')

colorbar = fig.colorbar(im, pad=0.2, aspect=15,)# ticks=[-0.3, 0, 0.3, 0.6])  # Set 3 ticks
colorbar.ax.set_position([ax.get_position().x1+0.02, ax.get_position().y0, 0.05, ax.get_position().height,])
colorbar.set_label(label="Novelty Modulation", rotation=270, labelpad=10)


In [ ]:
for state in ['active', 'passive']:
    for flash in ['nonshared_nonchange', 'change']:
        fig, ax = plt.subplots()
        fig.set_size_inches(10, 10)
        fig.suptitle(f'{state} {flash}')
        # condition_vals = vals[state][flash]
        vbn_utils.plot_comparison_matrix(*vals[1][state][flash].values(), ax=ax,labels=list(vals[1][state][flash].keys()), cmap='PiYG',colorbar=True, binarize=True, test_func=vbn_utils.permutation_test, corrected=True, nan_color='lightgray')

### Mousewise

In [ ]:
ctx_areas_ordered = ['VISall',]# 'VISp', 'VISl', 'VISrl', 'VISal', 'VISpm', 'VISam']
layers = ['2/3', '4', '5', '6']

layer_mousewise_noveltymod_vals = make_mousewise_nov_modulation_figure(ctx_areas_ordered, layers=layers, cell_types='all', clusters='all', units_subsample=units, 
                        flashes=['nonshared_nonchange', 'change'], return_vals=True, cell_count_thresh=5)

for il, layer in enumerate(layers):
    layer_mouse_count = np.sum(~np.isnan(layer_mousewise_noveltymod_vals['active']['change'][f'VISall_{layer}']))
    print(f'{area} {layer}: {layer_mouse_count} mice')

In [ ]:
plt.rcParams['font.size'] = 24
for state in ['active', 'passive']:
    for flash in ['nonshared_nonchange', 'change']:
        fig, ax = plt.subplots()
        # fig.set_size_inches(10, 10)
        fig.suptitle(f'{state} {flash}')
        vbn_utils.plot_comparison_matrix(*layer_mousewise_noveltymod_vals[state][flash].values(), ax=ax,labels=list(layer_mousewise_noveltymod_vals[state][flash].keys()), cmap='PiYG',colorbar=True, 
        test_func=vbn_utils.hybrid_permutation_test_2, binarize=True, corrected=True, nan_color='lightgray')

## Novelty modulation across cell types

In [ ]:
cell_type_color_dict = {'RS': 'teal',
                        'FS': 'red',
                        'SST': 'dodgerblue',
                        'VIP': 'mediumorchid'}

In [ ]:
cell_types = ['RS', 'FS', 'SST', 'VIP']
ctx_areas_ordered = ['VISp', 'VISl', 'VISrl', 'VISal', 'VISpm', 'VISam', 'VISall']
make_nov_mod_psth_figure('VISall', layers='all', cell_types=cell_types, clusters='all', norm=False, units_subsample=units, flashes=['nonshared_nonchange', 'change'])
# make_nov_mod_psth_figure(ctx_areas_ordered, layers='all', cell_types=cell_types, clusters='all', norm=False, state='active',
#                     flashes=['nonshared_nonchange', 'change'], plot_pvalues=False, smoothing_kernel_width=3, units_subsample=units)
# make_nov_modulation_figure('VISall', layers='all', cell_types=cell_types, clusters='all', 
#                             units_subsample=units, cell_type_color_dict=cell_type_color_dict,)

In [ ]:
ctx_areas_ordered = ['VISall', ['VISp', 'VISl', 'VISal'], ['VISrl', 'VISpm', 'VISam']]# 'VISp', 'VISl', 'VISrl', 'VISal', 'VISpm', 'VISam']
cell_types = ['RS', 'FS', 'SST', 'VIP']
vals = make_nov_modulation_figure(ctx_areas_ordered, layers='all', cell_types=cell_types, clusters='all', 
                    flashes=['nonshared_nonchange', 'change'], units_subsample=units, return_vals=True,)

In [ ]:
# save vals to pickle
import pickle
with open("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/novelty_modulation_across_celltypes.pkl", 'wb') as file:
    pickle.dump(vals, file)

In [ ]:
cmap.set_bad(color='gray')

val_matrix = np.full((len(ctx_areas_ordered), 4), np.nan)
for ia, area in enumerate(ctx_areas_ordered):
    for ic, cell_type in enumerate(cell_types):
        if not area=='VISall' and cell_type=='VIP':
            continue
        key = f'{area}_{cell_type}'
        val_matrix[ia, ic] = np.nanmedian(vals['active']['change'][key])

maxval = np.nanmax(np.abs(val_matrix))
norm = Normalize(vmin=-maxval, vmax=maxval)  # Normalize to the min and max of the matrix

fig, ax = plt.subplots()
im = ax.imshow(val_matrix.T, cmap=cmap, norm=norm, aspect='equal')
ax.set_yticks(np.arange(4))
ax.set_yticklabels(cell_types)
ax.set_xticks(np.arange(len(ctx_areas_ordered)))
ax.set_xticklabels(ctx_areas_ordered, rotation=90)
ax.set_xlabel('Area')
ax.set_ylabel('Cell Type')
# vbn_utils.formatFigure(fig, ax)
# plt.colorbar(im, label='Novelty Modulation Index')

colorbar = fig.colorbar(im, pad=0.2, aspect=15,)# ticks=[-0.3, 0, 0.3, 0.6])  # Set 3 ticks
colorbar.ax.set_position([ax.get_position().x1+0.02, ax.get_position().y0, 0.05, ax.get_position().height,])
colorbar.set_label(label="Novelty Modulation", rotation=270, labelpad=10)


In [ ]:
for state in ['active', 'passive']:
    for flash in ['nonshared_nonchange', 'change']:
        fig, ax = plt.subplots()
        fig.set_size_inches(10, 10)
        fig.suptitle(f'{state} {flash}')
        # condition_vals = vals[state][flash]
        vbn_utils.plot_comparison_matrix(*vals[1][state][flash].values(), ax=ax,labels=[k.replace('[', '').replace(']', '').replace("'", "").replace(', ', '_') for k in list(vals[1][state][flash].keys())], 
            cmap='PiYG',colorbar=True, test_func=vbn_utils.permutation_test, binarize=True, corrected=True, nan_color='lightgray')

### Mousewise

In [ ]:
ctx_areas_ordered = ['VISall',]# 'VISp', 'VISl', 'VISrl', 'VISal', 'VISpm', 'VISam']
# ctx_areas_ordered = ['VISall', ['VISp', 'VISl', 'VISal'], ['VISrl', 'VISpm', 'VISam'],]
cell_types = ['RS', 'FS', 'SST',]# 'VIP']
celltype_mousewise_noveltymod_vals = make_mousewise_nov_modulation_figure(ctx_areas_ordered, layers='all', cell_types=cell_types, clusters='all', 
                    flashes=['nonshared_nonchange', 'change'], units_subsample=units, return_vals=True,)

In [ ]:
fig, ax = plt.subplots()

for ic, cell_type in enumerate(cell_types):
    mod_inds = celltype_mousewise_noveltymod_vals['active']['change'][f'VISall_{cell_type}']
    median_val = np.nanmedian(mod_inds)
    ci = bootstrap_ci(mod_inds, np.nanmedian)
    # ci = np.percentile(mod_inds, [2.5, 97.5])
    error: list[list[Any]] = [[median_val-ci[0]], [ci[1]-median_val]]
    ax.plot(ic, median_val, 'o', color=cell_type_colors[cell_type], ms=14)
    ax.errorbar([ic], [median_val], yerr=error, color=cell_type_colors[cell_type], )
    print(np.sum(~np.isnan(mod_inds)))

ax.set_xticks(np.arange(len(cell_types)))
ax.set_xticklabels(cell_types, rotation=90)
vbn_utils.formatFigure(fig, ax, yLabel='Novelty Modulation')


In [ ]:
plt.rcParams['font.size'] = 24
for state in ['active', 'passive']:
    for flash in ['nonshared_nonchange', 'change']:
        fig, ax = plt.subplots()
        fig.suptitle(f'{state} {flash}')
        condition_vals = vals[state][flash]
        vbn_utils.plot_comparison_matrix(*celltype_mousewise_noveltymod_vals[state][flash].values(), ax=ax,labels=list(celltype_mousewise_noveltymod_vals[state][flash].keys()), cmap='PiYG',colorbar=True, 
            test_func=vbn_utils.hybrid_permutation_test_2, binarize=True, corrected=True, nan_color='lightgray')

In [ ]:
plt.rcParams['font.size'] = 20
make_novelty_vs_context_figure([['VISp', 'VISl', 'VISal'], ['VISrl', 'VISpm', 'VISam']], layers='all', cell_types=['RS', 'FS', 'SST'], clusters='all')
# make_nov_mod_psth_figure('VISall', layers='all', cell_types='all', clusters='all', norm=False, state=['active',],
#                         flashes=['nonshared_nonchange', 'shared_nonchange'], plot_pvalues=False, smoothing_kernel_width=3, units_subsample=units)

## Novelty modulation across training trajectories

In [ ]:
areas_ordered = ['VISall',]
make_nov_mod_psth_figure(areas_ordered, layers='all', cell_types='all', clusters='all', norm=False, state=['active',],
                        flashes=['nonshared_nonchange', 'change'], plot_pvalues=False, smoothing_kernel_width=3, units_subsample=ghg_units)
vals = make_nov_modulation_figure(areas_ordered, layers='all', cell_types='all', clusters='all', return_vals=True,
                        flashes=['nonshared_nonchange', 'change'], units_subsample=ghg_units, num_iterations=100)

## Running comparison across states

In [ ]:
passing_sessions = sessions_table[sessions_table['abnormal_activity'].isnull() & sessions_table['abnormal_histology'].isnull()]['ecephys_session_id'].values

familiar_running = stim_table[stim_table['engaged'] & (stim_table['experience_level']=='Familiar') & (stim_table['session_id'].isin(passing_sessions)) & (~stim_table['image_name'].isin(['im083_r', 'im111_r']))]['baseline_running']
novel_running = stim_table[stim_table['engaged'] & (stim_table['experience_level']=='Novel') & (stim_table['session_id'].isin(passing_sessions)) & (~stim_table['image_name'].isin(['im083_r', 'im111_r']))]['baseline_running']

plt.hist(familiar_running[~np.isnan(familiar_running)], density=True, alpha=0.5, bins=np.arange(0, 150, 10), color='b')
plt.hist(novel_running[~np.isnan(novel_running)], density=True, alpha=0.5, bins=np.arange(0, 150, 10), color='r')

print(scipy.stats.ranksums(familiar_running[~np.isnan(familiar_running)], novel_running[~np.isnan(novel_running)]))
print(np.nanmean(familiar_running), np.nanmean(novel_running))

## Putting all modulation indices together

In [ ]:
dfs = []
division_to_labels = {
    'areas': ['VISall', 'VISp', 'VISl', 'VISrl', 'VISal', 'VISpm', 'VISam', 'LGd', 'LP', 'SCMRN'],
    'layers': [area +'_' + layer for area in ['VISall', 'VISp', 'VISl', 'VISrl', 'VISal', 'VISpm', 'VISam'] for layer in ['2/3', '4', '5', '6']],
    'celltypes': [area +'_' + celltype for area in ['VISall', 'VISp', 'VISl', 'VISrl', 'VISal', 'VISpm', 'VISam'] for celltype in ['RS', 'FS', 'SST', 'VIP']]
}

for division in ['areas', 'layers', 'celltypes']:
    # load change mod pickle file
    with open(f"/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/change_modulation_across_{division}.pkl", 'rb') as file:
        change_vals = pickle.load(file)
    
    # load state mod pickle file
    with open(f"/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/state_modulation_across_{division}.pkl", 'rb') as file:
        state_vals = pickle.load(file)
    
    # load novelty mod pickle file
    with open(f"/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/novelty_modulation_across_{division}.pkl", 'rb') as file:
        novelty_vals = pickle.load(file)
    

    # create a DataFrame for each division
    df = {}
    for row in division_to_labels[division]:
        df[row] = {
            'change_modulation': np.nanmedian(change_vals['active']['mod_index_norm'][row]),
            'state_modulation': np.nanmedian(state_vals['change']['mod_index_norm'][row]),
            'novelty_modulation': np.nanmedian(novelty_vals['active']['change'][row])
        }

    df = pd.DataFrame(df).T
    dfs.append(df)

In [ ]:
concat = pd.concat(dfs)
concat

In [ ]:
concat.to_csv("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/change_state_novelty_modulation_across_areas_layers_celltypes.csv")

## Comparison of change modulation and novelty modulation

In [ ]:
unit_ids = vbn_utils.get_unit_ids(units, 'VISall', clusters='all', experience='Novel')

active_change_mods = np.array([modulation_summary[u]['shared_active'] for u in unit_ids])
passive_change_mods = np.array([modulation_summary[u]['change_active'] for u in unit_ids])

nonan = (~np.isnan(active_change_mods)) & (~np.isnan(passive_change_mods))
print(np.sum(nonan)/len(nonan))
plt.plot(active_change_mods, passive_change_mods, 'k.', alpha=0.1)
np.corrcoef(active_change_mods[nonan], passive_change_mods[nonan])

## Sparseness

In [ ]:
import warnings
warnings.filterwarnings('ignore')

areas = ['VISp', 'VISl', 'VISal', 'VISrl', 'VISam', 'VISpm', 'LGd', 'LP', 'VISall', 'SCMRN', 'Hipp']
cell_type = 'RS'
layer = 'all'
cluster = 'all'

window_duration = 40
window_starts = np.arange(0, 400)
window_ends = window_starts + window_duration
bins = window_starts + window_duration/2 - 50

for area in areas:
    fig, ax = plt.subplots()
    fig.suptitle(f'{area}')
    #fig.set_size_inches(12,4)
    #ax[0].set_title(area)
    unitIDs = get_unit_ids(units, area, cell_type, layer, cluster)
    base_slice = slice(0, 50)
    base_sub = lambda x: x - x[:, base_slice].mean(axis=1)[:, None]
    
    num_sessions = []
    all_sparseness = []
    for experience, color in zip(['Familiar', 'Novel'], ['b', 'r']):
        session_ids = units[units['experience_level']==experience]['ecephys_session_id'].unique()
        exp_sparseness = []
        for session_id in session_ids:
            sess_sparseness = []
            session_units = units[units['ecephys_session_id']==session_id]['unit_id'].values
            sess_unitIDs = [u for u in unitIDs if u in session_units]
            if len(sess_unitIDs)==0:
                continue
            data = np.stack([flash_data[u][experience]['active']['nonchange_nolick'] for u in sess_unitIDs])*1000
            data_base_sub = base_sub(data)
            for slice_start, slice_end in zip(window_starts, window_ends):
                
                resp_vect = data_base_sub[:, slice_start:slice_end].mean(axis=1)
                sparseness = calc_pop_sparseness_kurtosis(resp_vect)
                #sparseness = calc_pop_sparseness(resp_vect)
                sess_sparseness.append(sparseness)
            if len(sess_sparseness)>0:
                exp_sparseness.append(sess_sparseness)

        mean_sem_plot(np.array(exp_sparseness), ax, color=color, x=bins)
        num_sessions.append(len(exp_sparseness))
        all_sparseness.append(np.array(exp_sparseness))
    
    formatFigure(fig, ax)
    ax.set_title(f'fam: {num_sessions[0]},  nov: {num_sessions[1]}')
    pvals = [scipy.stats.ranksums(ftimepoint, ntimepoint)[1] for ftimepoint, ntimepoint in zip(all_sparseness[0].T, all_sparseness[1].T)]
    corrected = multiple_comparisons(pvals)
    ax.plot(bins[corrected[0]], [ax.get_ylim()[1]]*np.sum(corrected[0]), 'k.')
        #ax.plot(bins, exp_sparseness, color=color)

    #     mean_sem_plot(data_base_sub[:, 50:300], ax[0], color=color)
        
    #     # vals, chist = cumulative_hist(mean_resps)
    #     # ax.plot(vals, chist)
    #     ax[iresp+1].hist(mean_resps, bins = np.arange(-50, 75, 3), alpha = 1, color=color, density=True, histtype='step')
    #     ax[iresp+1].set_yscale('log')
        
    #     #print(np.nanmedian(mean_resps))

    # pval = scipy.stats.kstest(*means, nan_policy='omit')
    # ax[iresp+1].set_title(f'kstest p: {pval[1]} \n fam: {np.sum(~np.isnan(means[0]))}, nov: {np.sum(~np.isnan(means[1]))}')
    # ylim = ax[iresp+1].get_ylim()[1]
    # [ax[iresp+1].plot(np.nanmedian(mrs), ylim, 'v', c=color) for mrs, color in zip(means, ['b', 'r'])]

    # [formatFigure(fig, a) for a in ax]
    # ax[0].axvspan(20, 70, 0.95, 1, color='0.5', alpha=0.5)
    # ax[0].axvspan(70, 120, 0.95, 1, color='0.6', alpha=0.5)
    # fig.savefig(os.path.join("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/figures/fam_vs_nov_timecourse", f'{area}_{cell_type}_{layer}_{cluster}_nov_fam.pdf'))

# Recompute change/state modulation with running-matched trials

In [ ]:
active_passive_change_baseline_running_matched_trial_inds_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/active_passive_baseline_running_matches_nonshared_change_prechange.pkl"
import pickle
with open(active_passive_change_baseline_running_matched_trial_inds_file, 'rb') as f:
    active_passive_nonshared_change_prechange_baseline_running_matched_trial_inds = pickle.load(f)

In [ ]:
stim_table['active_change_baseline_running_matched'] = False
stim_table.loc[active_passive_nonshared_change_prechange_baseline_running_matched_trial_inds['active']['change'], 'active_change_baseline_running_matched'] = True

stim_table['active_prechange_baseline_running_matched'] = False
stim_table.loc[active_passive_nonshared_change_prechange_baseline_running_matched_trial_inds['active']['prechange'], 'active_prechange_baseline_running_matched'] = True

stim_table['passive_change_baseline_running_matched'] = False
stim_table.loc[active_passive_nonshared_change_prechange_baseline_running_matched_trial_inds['passive']['change'], 'passive_change_baseline_running_matched'] = True

stim_table['passive_prechange_baseline_running_matched'] = False
stim_table.loc[active_passive_nonshared_change_prechange_baseline_running_matched_trial_inds['passive']['prechange'], 'passive_prechange_baseline_running_matched'] = True

In [ ]:
unit_filter = du.apply_unit_quality_filter(units)
unit_ids = units.loc[unit_filter]['unit_id'].values

flash_data_run_matched = {uid: {cond: {stims: [] for stims in ['change', 'prechange','shared_nonchange', 'nonshared_nonchange']} for cond in ['active', 'passive']} for uid in unit_ids}



session_list = [str(sess) for sess in np.unique(units.loc[unit_filter]['ecephys_session_id'])]
for cond, tensor_to_use in zip(['active', 'passive'], [active_tensor_file, passive_tensor_file]):


    #for non shared images
    change_data, prechange_data, unitIDs = vbn_utils.change_prechange_matched_psth(tensor_to_use, 
                                stim_table, 
                                session_list, 
                                unit_ids, 
                                baseline_length=50, 
                                resp_window_length=750,
                                comparison='change_prechange',
                                match_running_speed=True,
                                match_running_col_prefix=cond+'_')

    changes = np.concatenate([c for c in change_data if c.size>1])
    prechanges = np.concatenate([pc for pc,c in zip(prechange_data, change_data) if c.size>1])
    unitIDs = np.concatenate([u for u, c in zip(unitIDs, change_data) if c.size>1])

    for cdata, predata, uid in zip(changes, prechanges, unitIDs):
        flash_data_run_matched[uid][cond]['change'] = cdata
        flash_data_run_matched[uid][cond]['prechange'] = predata

In [ ]:
import pickle

with open("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/change_prechange_responses_active_passive_run_matched_by_unitid.pkl", 'wb') as file:
    pickle.dump(flash_data_run_matched, file)

# with open("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/change_prechange_trial_by_trial_responses_active_passive_by_unitid.pkl", 'wb') as file:
#     pickle.dump(trial_data, file)

### Load pre-computed running-matched data

In [ ]:
import pickle

with open("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/change_prechange_responses_active_passive_run_matched_by_unitid.pkl", 'rb') as file:
    flash_data_run_matched = pickle.load(file)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

areas = ['VISp', 'VISl', 'VISal', 'VISrl', 'VISpm', 'VISam', 'VISall', 'LGd', 'LP', 'Hipp', 'SCMRN']
layers = ['2/3', '4', '5', '6', 'all']

base_slice = slice(0, 50)
resp_slice = slice(70, 150)
binsize = 5
time = np.arange(-50, 750, binsize)

base_funcs = [base_sub, no_base_sub]

unitIDs = units.loc[du.apply_unit_quality_filter(units)]['unit_id'].values

response_summary_run_matched = {u:{base_func: {cond: {flash: [] for flash in ['change', 'prechange',]} for cond in ['active', 'passive']} for base_func in base_funcs} for u in unitIDs}

for base_func in base_funcs:
    # for cond, linestyle in zip(['active', 'passive'], ['solid', 'dotted']):
    for cond in ['active', 'passive']:
        for flash in ['change', 'prechange',]:
            for uu in flash_data_run_matched.keys():
                if len(flash_data_run_matched[uu][cond][flash])==0:
                    flash_data_run_matched[uu][cond][flash] = np.full((800), np.nan)
            cdata = base_func(np.stack([flash_data_run_matched[u][cond][flash] for u in unitIDs])*1000)
            cresponses = np.mean(cdata[:, resp_slice], axis=1)

            for uid, u in enumerate(unitIDs):
                response_summary_run_matched[u][base_func][cond][flash] = cresponses[uid]


### Across areas

In [ ]:
plt.rcParams['font.size'] = 18
areas_ordered = ['LGd', 'LP', 'VISp', 'VISl', 'VISrl', 'VISal', 'VISpm', 'VISam', 'Hipp', 'SCMRN']
make_psth_figure(areas_ordered, layers='all', cell_types='all', clusters='sensory', comparison='change', unit_subsample=units,
                 norm=False, split=False, use_ccf_color=False, smooth_kernel_width=3, base_sub_func=base_sub, flash_data=flash_data_run_matched)

In [ ]:
areas_ordered = ['LGd', 'LP', 'VISp', 'VISl', 'VISrl', 'VISal', 'VISpm', 'VISam', 'SCMRN']

change_vals = make_modulation_figure(areas_ordered, layers='all', cell_types='all', clusters='sensory', comparison='change', 
                metrics=['mod_index_norm',], plot_comparison_mat=True, return_vals=True, num_iterations=10, base_sub_func=base_sub, response_summary=response_summary_run_matched,
                unit_subsample=units)
state_vals = make_modulation_figure(areas_ordered, layers='all', cell_types='all', clusters='sensory', comparison='state', 
                metrics=['mod_index_norm',], plot_comparison_mat=True, return_vals=True, num_iterations=10, base_sub_func=base_sub, response_summary=response_summary_run_matched,
                unit_subsample=units)

fig, ax = plt.subplots()
for ia, area in enumerate(areas_ordered):
    color = ccf_utils.get_area_color(area, structure_tree)
    #fcs = [color, 'w']
    fcs = ['w',]
    # for ivals, vals in enumerate([change_vals['active']['mod_index_norm'][area], state_vals['change']['mod_index_norm'][area]]):
    for ivals, vals in enumerate([state_vals['change']['mod_index_norm'][area],]):


        median_val = np.nanmedian(vals)
        # ax.plot(ia, median_val, 'o', color=ccf_utils.get_area_color(area, structure_tree))
        ci = bootstrap_ci(vals, np.median, num_iterations=1000)
        error = [[median_val-ci[0]], [ci[1]-median_val]]
        ax.plot(ia, median_val, 'o', mec=color, mfc=fcs[ivals], ms=12, markeredgewidth=2)
        ax.errorbar([ia], [median_val], yerr=error, color=color)

ax.axhline(0, ls='dotted', color='k')
ax.set_xticks(np.arange(len(areas_ordered)))
ax.set_xticklabels(areas_ordered[:-1] + ['SCm/MRN'], rotation=45,)
ax.tick_params(axis='both', labelsize=20)  # Change font size for both x and y axes
ax.set_ylabel('Modulation Index', fontsize=20)
for spine in ax.spines.values():
    spine.set_linewidth(1.5)  # Set spine line width to 2 points

# Increase the line width of ticks
ax.tick_params(width=1.5)  # Set tick line width to 2 points

plt.rcParams['font.size'] = 24
vbn_utils.formatFigure(fig, ax)

In [ ]:
fig, ax = plt.subplots()
# fig.suptitle(f'{state} {flash}')
# condition_vals = vals[state][flash]
vbn_utils.plot_comparison_matrix(*state_vals['change']['mod_index_norm'].values(), ax=ax,labels=list(state_vals['change']['mod_index_norm'].keys()), cmap='PiYG',colorbar=True, corrected=True, binarize=True, nan_color='lightgray')

### Across layers

In [ ]:
make_psth_figure('VISall', layers=['2/3', '4', '5', '6'], cell_types='all', clusters='sensory', 
                comparison='change', norm=False, split=False, use_ccf_color=False,
                unit_subsample=units, base_sub_func=base_sub, flash_data=flash_data_run_matched)


In [ ]:
layers=['2/3', '4', '5', '6']

regions = ['VISall', 'VISp', 'VISl', 'VISrl', 'VISal', 'VISpm', 'VISam']

change_vals = make_modulation_figure(regions, layers=layers, cell_types='all', clusters='sensory', comparison='change', 
                metrics=['mod_index_norm',], plot_comparison_mat=True, return_vals=True, num_iterations=10, response_summary=response_summary_run_matched)
state_vals = make_modulation_figure(regions, layers=layers, cell_types='all', clusters='sensory', comparison='state', 
                metrics=['mod_index_norm',], plot_comparison_mat=True, return_vals=True, num_iterations=10, response_summary=response_summary_run_matched)

In [ ]:
layers=['2/3', '4', '5', '6']

change_vals = make_modulation_figure('VISall', layers=layers, cell_types='all', clusters='sensory', comparison='change', 
                metrics=['mod_index_norm',], plot_comparison_mat=True, return_vals=True, num_iterations=10, response_summary=response_summary_run_matched)
state_vals = make_modulation_figure('VISall', layers=layers, cell_types='all', clusters='sensory', comparison='state', 
                metrics=['mod_index_norm',], plot_comparison_mat=True, return_vals=True, num_iterations=10, response_summary=response_summary_run_matched)

fig, ax = plt.subplots()
alphas = [1, 0.75, 0.5, 0.35]
for il, layer in enumerate(layers):
    color = ccf_utils.get_area_color('VISall', structure_tree)
    fcs = [color, 'w']
    fcs = ['w',]
    # for ivals, vals in enumerate([change_vals['active']['mod_index_norm']['VISall_'+layer], 
    #                                 state_vals['change']['mod_index_norm']['VISall_'+layer]]):
    for ivals, vals in enumerate([state_vals['change']['mod_index_norm']['VISall_'+layer],]):

        median_val = np.nanmedian(vals)
        # ax.plot(ia, median_val, 'o', color=ccf_utils.get_area_color(area, structure_tree))
        ci = bootstrap_ci(vals, np.median, num_iterations=1000)
        error = [[median_val-ci[0]], [ci[1]-median_val]]
        ax.plot(il, median_val, 'o', mec=color, mfc=fcs[ivals], ms=12, alpha=alphas[il], markeredgewidth=3)
        ax.errorbar([il], [median_val], yerr=error, color=color, alpha=alphas[il])

ax.axhline(0, ls='dotted', color='k')
ax.set_xticks(np.arange(len(layers)))
ax.set_xticklabels(layers,)
ax.tick_params(axis='both', labelsize=20)  # Change font size for both x and y axes
ax.set_ylabel('Modulation Index', fontsize=20)
for spine in ax.spines.values():
    spine.set_linewidth(1.5)  # Set spine line width to 2 points

# Increase the line width of ticks
ax.tick_params(width=1.5)  # Set tick line width to 2 points
plt.rcParams['font.size'] = 24
vbn_utils.formatFigure(fig, ax)

In [ ]:
fig, ax = plt.subplots()
vbn_utils.plot_comparison_matrix(*state_vals['change']['mod_index_norm'].values(), ax=ax,labels=list(state_vals['change']['mod_index_norm'].keys()), cmap='PiYG',colorbar=True, corrected=True, binarize=True, nan_color='lightgray')

### Across cell types

In [ ]:
make_psth_figure('VISall', layers='all', cell_types=['RS', 'FS', 'SST', 'VIP'], clusters='sensory', 
                comparison='change', norm=False, split=False, use_ccf_color=False, smooth_kernel_width=3,
                base_sub_func=no_base_sub, unit_subsample=low_baseline_drift_units, flash_data=flash_data_run_matched)

In [ ]:
cell_types=['RS', 'FS', 'SST', 'VIP']
cell_type_colors = {'RS': 'teal', 
                    'FS': 'red',
                    'SST': 'dodgerblue',
                    'VIP': 'mediumorchid'}

regions = ['VISall', 'VISp', 'VISl', 'VISrl', 'VISal', 'VISpm', 'VISam']

change_vals = make_modulation_figure(regions, layers='all', cell_types=cell_types, clusters='sensory', comparison='change', 
                metrics=['mod_index_norm',], plot_comparison_mat=True, return_vals=True, num_iterations=10, response_summary=response_summary_run_matched,)
state_vals = make_modulation_figure(regions, layers='all', cell_types=cell_types, clusters='sensory', comparison='state', 
                metrics=['mod_index_norm',], plot_comparison_mat=True, return_vals=True, num_iterations=10, response_summary=response_summary_run_matched,)

In [ ]:
cell_types=['RS', 'FS', 'SST', 'VIP']
cell_type_colors = {'RS': 'teal', 
                    'FS': 'red',
                    'SST': 'dodgerblue',
                    'VIP': 'mediumorchid'}

change_vals = make_modulation_figure('VISall', layers='all', cell_types=cell_types, clusters='sensory', comparison='change', 
                metrics=['mod_index_norm',], plot_comparison_mat=True, return_vals=True, num_iterations=10, response_summary=response_summary_run_matched, base_sub_func=base_sub,
                unit_subsample=units)
state_vals = make_modulation_figure('VISall', layers='all', cell_types=cell_types, clusters='sensory', comparison='state', 
                metrics=['mod_index_norm',], plot_comparison_mat=True, return_vals=True, num_iterations=10, response_summary=response_summary_run_matched, base_sub_func=base_sub,
                unit_subsample=units)

fig, ax = plt.subplots()
alphas = [1, 0.75, 0.5, 0.35]
for ic, cell_type in enumerate(cell_types):
    color = cell_type_colors[cell_type]
    # fcs = [color, 'w']
    fcs = ['w',]
    # for ivals, vals in enumerate([change_vals['active']['mod_index_norm']['VISall_'+cell_type],
    #                                 state_vals['change']['mod_index_norm']['VISall_'+cell_type]]):
    for ivals, vals in enumerate([state_vals['change']['mod_index_norm']['VISall_'+cell_type],]):

        median_val = np.nanmedian(vals)
        # ax.plot(ia, median_val, 'o', color=ccf_utils.get_area_color(area, structure_tree))
        ci = bootstrap_ci(vals, np.median, num_iterations=1000)
        error = [[median_val-ci[0]], [ci[1]-median_val]]
        ax.plot(ic, median_val, 'o', mec=color, mfc=fcs[ivals], ms=12, markeredgewidth=2)
        ax.errorbar([ic], [median_val], yerr=error, color=color)
ax.axhline(0, ls='dotted', color='k')
ax.set_xticks(np.arange(len(cell_types)))
ax.set_xticklabels(cell_types,)
ax.tick_params(axis='both', labelsize=20)  # Change font size for both x and y axes
ax.set_ylabel('Modulation Index', fontsize=20)
for spine in ax.spines.values():
    spine.set_linewidth(1.5)  # Set spine line width to 2 points

# Increase the line width of ticks
ax.tick_params(width=1.5)  # Set tick line width to 2 points
plt.rcParams['font.size'] = 40
vbn_utils.formatFigure(fig, ax)

In [ ]:
plt.rcParams['font.size'] = 22
fig, ax = plt.subplots()
vbn_utils.plot_comparison_matrix(*state_vals['change']['mod_index_norm'].values(), ax=ax,labels=list(state_vals['change']['mod_index_norm'].keys()), cmap='PiYG',colorbar=True, corrected=True, binarize=True, nan_color='lightgray')